# Snekmer Learn/Apply — User Tutorial

This notebook runs the full **Learn/Apply** pipeline using `snekmer easy` and shows you how to read, filter, and visualize the results.

**You do not need to understand the internal pipeline to use this notebook.** All processing is done by a single command; this notebook focuses on working with the output.

> For a step-by-step walkthrough of every internal pipeline rule (vectorization, k-mer counts, confidence calibration), see `snekmer_learn_apply_tutorial.ipynb` in this directory.

**What you need:**
- Snekmer installed and activated (see the [installation guide](https://snekmer.readthedocs.io/en/latest/getting_started/install.html))
- Training FASTA files with known family annotations
- Query FASTA file(s) to annotate
- An annotation file (`.ann`) **or** family labels embedded in FASTA headers

**Demo data** used in this notebook is included in the Snekmer repository:
```
resources/demo_sequences/learn_apply_inputs/
├── learn/         ← 10 training FASTA files (10,000 proteins, 200 TIGRFAM families)
├── apply/         ← 1 query FASTA (3,000 proteins)
└── annotations/   ← TIGRFAMs_annotation.ann
```


## Workflow

Snekmer's **Learn/Apply** pipeline (bottom row below) takes annotated training sequences, builds k-mer count vectors, fits a per-family confidence model, then scores query sequences against that model via cosine similarity.

![Snekmer workflow overview](https://raw.githubusercontent.com/PNNL-CompBio/Snekmer/main/resources/images/snekmer_workflow.svg)

## Setup

Register your Snekmer virtual environment as a Jupyter kernel (one-time step):

```bash
source ~/snekmer_env/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=snekmer
jupyter notebook
```

> **Note:** This notebook assumes you are running it from the `resources/tutorial/` directory.


In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

# Paths relative to resources/tutorial/
DEMO_ROOT  = Path("../demo_sequences/learn_apply_inputs")
TRAIN_DIR  = DEMO_ROOT / "learn"
QUERY_FILE = DEMO_ROOT / "apply" / "test_sequences_1.fasta"
ANN_FILE   = DEMO_ROOT / "annotations" / "TIGRFAMs_annotation.ann"
OUTPUT_DIR = Path("easy_output")
RESULTS    = OUTPUT_DIR / "apply" / "snekmer_results.csv"

# Verify demo data is present
for p in [TRAIN_DIR, QUERY_FILE, ANN_FILE]:
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {p}")


## Running the pipeline

`easy` takes your training sequences, query sequences, and annotation file and runs the full Learn → Apply pipeline in one command. All parameters have sensible defaults.

**Key parameters** (all optional — defaults work for most datasets):

| Flag | Default | Description |
|---|---|---|
| `--k` | `8` | K-mer length |
| `--alphabet` | `2` (solvacc) | Amino acid reduction alphabet (0–5, see `--help`) |
| `--selection` | `top_hit` | Prediction selection method |
| `--threshold` | `Median` | Score threshold for family filtering |
| `--cores` | all CPUs | CPU cores to use |
| `--fragmentation` | off | Split training sequences into fragments before k-merizing |
| `--copy-files` | off | Copy inputs into workspace instead of symlinking |
| `--dry-run` | off | Preview pipeline steps without running |


In [ ]:
# Run easy — replace paths with your own data to annotate real sequences
result = subprocess.run(
    [
        sys.executable, "-m", "snekmer",  # or just "snekmer" if on PATH
        "easy",
        "--train",  str(TRAIN_DIR),
        "--query",  str(QUERY_FILE),
        "--ann",    str(ANN_FILE),
        "--output", str(OUTPUT_DIR),
        "--cores",  "1",
    ],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print("STDERR:\n", result.stderr[-3000:])
    raise RuntimeError("easy failed")
print("Pipeline complete. Results at:", RESULTS)


## Reading the results

The main output is `apply/snekmer_results.csv` — one row per query sequence.

| Column | Description |
|---|---|
| `Sequence` | Sequence identifier from the FASTA header |
| `Prediction` | Predicted family (highest cosine similarity to training profiles) |
| `Score` | Cosine similarity (0–1). **Score = 0 means no shared k-mers — exclude these.** |
| `delta` | Gap between top and second-best score. Larger = more unambiguous prediction. |
| `Confidence` | Calibrated probability the prediction is correct (0–1) |


In [ ]:
df = pd.read_csv(RESULTS)
print(f"Total query sequences: {len(df)}")
df.head(10)


## Filtering by confidence

All sequences receive a prediction, but not all predictions are reliable. Use **Confidence** and **Score** to filter:

- **Confidence ≥ 0.95** — good starting threshold for high-quality annotations
- **Score > 0** — removes sequences with no shared k-mers (no real signal)

Adjust the threshold to trade precision for recall:
- Higher (e.g. 0.99) → fewer predictions, higher reliability
- Lower (e.g. 0.80) → more predictions, more false positives


In [ ]:
CONF_THRESHOLD = 0.95

high_conf = df[(df["Confidence"] >= CONF_THRESHOLD) & (df["Score"] > 0)].copy()
print(f"High-confidence predictions (≥{CONF_THRESHOLD}): {len(high_conf)} / {len(df)}")
high_conf.to_csv(OUTPUT_DIR / "high_confidence_results.csv", index=False)
print(f"Saved to: {OUTPUT_DIR / 'high_confidence_results.csv'}")
high_conf.head(10)


## Visualizing predictions


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Top predicted families
top_fam = high_conf["Prediction"].value_counts().head(15)
top_fam.plot(kind="bar", ax=axes[0], color="#2166ac")
axes[0].set_title(f"Top 15 predicted families (confidence ≥ {CONF_THRESHOLD})")
axes[0].set_xlabel("Family")
axes[0].set_ylabel("Sequences")
axes[0].tick_params(axis="x", rotation=45)

# Confidence distribution
axes[1].hist(df["Confidence"], bins=30, color="#4dac26", edgecolor="white")
axes[1].axvline(CONF_THRESHOLD, color="#d01c8b", linestyle="--", label=f"Threshold ({CONF_THRESHOLD})")
axes[1].set_title("Confidence score distribution (all sequences)")
axes[1].set_xlabel("Confidence")
axes[1].set_ylabel("Sequences")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Score = 0 (no signal): {(df['Score'] == 0).sum()} sequences")


## Score vs. Confidence scatter

Plotting Score against Confidence helps identify the reliable prediction space. Sequences in the top-right quadrant (high score, high confidence) are the strongest annotations.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(df["Score"], df["Confidence"], alpha=0.25, s=6,
                c=df["delta"], cmap="viridis")
ax.axhline(CONF_THRESHOLD, color="#d01c8b", linestyle="--", linewidth=1,
           label=f"Confidence = {CONF_THRESHOLD}")
ax.axvline(0, color="gray", linestyle=":", linewidth=0.8)
plt.colorbar(sc, ax=ax, label="delta (score gap)")
ax.set_xlabel("Score (cosine similarity)")
ax.set_ylabel("Confidence")
ax.set_title("Score vs. Confidence — colored by delta")
ax.legend()
plt.tight_layout()
plt.show()


## Post-hoc evaluation (when ground truth is available)

When you know the true family labels for your query sequences (as with this demo dataset), you can assess prediction quality by comparing Snekmer's output to the ground truth.

The demo test set has 3,000 proteins across three groups:
- **In-family** — from TIGRFAM families present in the training set
- **Other annotated** — from families *not* in the training set
- **Unannotated** — no known family assignment

> **Note:** With this small training set (200 families, 10,000 sequences), accuracy represents a lower bound. Performance improves substantially with larger, more diverse training data.


In [ ]:
conf_cutoff, score_cutoff, delta_cutoff = CONF_THRESHOLD, None, None

edf = pd.read_csv(RESULTS)
edf.columns = edf.columns.str.strip().str.capitalize()
edf["Accession"] = edf["Sequence"].str.split("|").str[1].fillna(edf["Sequence"])

ann_gt = pd.read_csv(ANN_FILE, sep="\t").rename(
    columns={"id": "Accession", "family": "Truefamily"}
)
edf = edf.merge(ann_gt, on="Accession", how="left")

kept = pd.Series(True, index=edf.index)
for col, cut in [("Confidence", conf_cutoff), ("Score", score_cutoff), ("Delta", delta_cutoff)]:
    if cut is not None:
        kept &= edf[col] >= cut

known = edf["Truefamily"].notna()
corr  = known & (edf["Prediction"] == edf["Truefamily"])
counts = {
    "True Positive":       int((kept & corr).sum()),
    "False Positive":      int((kept & known & ~corr).sum()),
    "Filtered (Known)":    int((~kept & known).sum()),
    "Predicted (Unknown)": int((~known & kept).sum()),
    "Filtered (Unknown)":  int((~known & ~kept).sum()),
}
TP, FP, FK = counts["True Positive"], counts["False Positive"], counts["Filtered (Known)"]
prec = TP / (TP + FP) if TP + FP else float("nan")
rec  = TP / (TP + FK) if TP + FK else float("nan")
print(f"Kept: {kept.sum()}/{len(edf)}  |  TP:{TP}  FP:{FP}  Filtered(Known):{FK}")
print(f"Precision: {prec:.3f}   Recall: {rec:.3f}")

colors = {
    "True Positive":       "#1b9e77",
    "False Positive":      "#d95f02",
    "Filtered (Known)":    "#757575",
    "Predicted (Unknown)": "#4575b4",
    "Filtered (Unknown)":  "#bdbdbd",
}
fig, ax = plt.subplots(figsize=(5.5, 3.5))
for group, keys in [
    ("Known",   ["True Positive", "False Positive", "Filtered (Known)"]),
    ("Unknown", ["Predicted (Unknown)", "Filtered (Unknown)"]),
]:
    bot = 0
    for k in keys:
        ax.bar(group, counts[k], bottom=bot, color=colors[k], label=k)
        bot += counts[k]
ax.set(title="Annotation prediction results", ylabel="Sequences")
h, l = ax.get_legend_handles_labels()
ax.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys(), ncol=2, fontsize=8)
plt.tight_layout()
plt.show()


## Next steps

- **Your own data:** Replace `TRAIN_DIR`, `QUERY_FILE`, and `ANN_FILE` with paths to your sequences and run the pipeline cell again.
- **Adjust parameters:** Pass `--k`, `--alphabet`, or `--selection` flags in the `subprocess.run` call to tune the model.
- **Additive training:** To build on an existing model with new sequences, use `snekmer learn` / `snekmer apply` directly — see the [full pipeline reference](https://snekmer.readthedocs.io/en/latest/tutorial/snekmer_learnapp_tutorial.html).
- **Understand the internals:** Open `snekmer_learn_apply_rules_reference.ipynb` in this directory for a rule-by-rule walkthrough of every computation.
